# DAEN 328 Assignment 6 Guide: Docker and Compose

This guide teaches every skill Assignment 6 needs. Read it next to **Assignment06_Instructions.pdf**, which lists exactly what is graded and what to submit.

**How the guide maps to the assignment**

| Guide section | Assignment part |
|---|---|
| 1 to 2: what Docker is, install and check | needed for everything |
| 3 to 5: first containers, your first image, the quotes app with Compose | **Part A** (45 points) |
| 6: running your own Assignment 5 code in containers | **Part B** (45 points) and **Part C** |
| 7 to 8: troubleshooting and a command cheat sheet | everything |

**Optional:** `Assignment06_Example_Three_Services.ipynb` runs a small, complete version of the Part B setup on toy data (quotes). You do not have to use it. Open it if Part B is not working and you want to see the same pattern working first.

Every Docker command in this guide runs in a **terminal** (Terminal on macOS, PowerShell on Windows), not in a notebook cell. Keep this notebook open as reading material and type the commands in your terminal.

Wherever you see **Screenshot for Part A** or **Screenshot for Part B**, take that screenshot right then. Each one goes in your PDF.

> **Install Docker Desktop before Thursday's class** (Section 2). On some Windows laptops the install needs a restart and a BIOS setting, and that is much easier to fix with help in class than at 11 pm.

# 1. What is Docker?

**Docker** packages an application **and everything it needs** (Python, libraries, system tools, settings) into one unit called a **container**. The container runs the same way on your laptop, a classmate's laptop, or a cloud server.

This solves the oldest problem in software: *"it works on my machine."* In Assignment 5 your loader and dashboard only work on a laptop that has the right Python, the right packages, a local PostgreSQL with your password, and the 7 tables you created by hand in pgAdmin. After this assignment, anyone with Docker can run your whole pipeline with one command.

### Three words you will use all the time

| Word | What it is | Analogy |
|---|---|---|
| **Image** | A read-only template: an operating system layer, Python, your packages, your code | A class in Python |
| **Container** | A running instance of an image. You can run many containers from one image | An object created from the class |
| **Dockerfile** | A text file of instructions that builds an image | The recipe |

### Containers vs virtual machines

| | Containers | Virtual machines |
|---|---|---|
| Start time | Seconds | Minutes |
| Size | Megabytes | Gigabytes |
| What they contain | Your app and its libraries; they share the host's kernel | A full operating system |

A virtual machine is a whole apartment with its own kitchen and plumbing. A container is a bedroom in a shared apartment: private, but sharing the building's plumbing.

Further reading: [Docker overview](https://docs.docker.com/get-started/overview/), [What is a container?](https://www.docker.com/resources/what-container)

# 2. Install Docker Desktop and check it works

1. Go to the [Docker Desktop download page](https://docs.docker.com/get-started/get-docker/) and pick your system. On a Mac, choose **Apple silicon** (M1 to M4) or **Intel** to match your laptop (Apple menu, About This Mac).
2. Run the installer with the default options.
3. Restart your computer.
4. Open Docker Desktop and wait until it says **Engine running** (bottom left).

Then, in a terminal:

```bash
docker --version
docker compose version
```

You should see two version lines, for example `Docker version 29.x` and `Docker Compose version v2.x` or newer.

> **`docker compose` (with a space) is the current command.** Older tutorials, and some answers you find online, use `docker-compose` with a hyphen. That is the retired version 1 tool. Use the space form for everything in this course.

## Troubleshooting the install

**Windows**
- **Stuck on "Starting..."**: quit Docker Desktop from the system tray and open it again; if that fails, restart the laptop.
- **WSL 2 errors**: Docker on Windows runs on WSL 2. In PowerShell run `wsl --update`, then `wsl --list --verbose`. If your distribution shows version 1, run `wsl --set-version <distro-name> 2`. Guide: [Docker Desktop WSL 2 backend](https://docs.docker.com/desktop/features/wsl/).
- **"Virtualization not enabled"**: open Task Manager, Performance tab, CPU, and check "Virtualization". If it says Disabled, turn it on in your BIOS (search "enable virtualization" plus your laptop model).
- **Last resort**: Docker Desktop, Troubleshoot (bug icon), **Reset to factory defaults**.

**macOS**
- **Docker will not start**: quit it, force quit any Docker processes in Activity Monitor, and open it again.
- **Wrong chip version**: an Intel build on an Apple silicon Mac (or the reverse) will not start. Reinstall the right one.
- **Diagnostics**: Docker whale icon in the menu bar, Troubleshoot, Run diagnostics.

More help: [Docker Desktop troubleshooting](https://docs.docker.com/desktop/troubleshoot-and-support/troubleshoot/)

# 3. Your first containers

## 3.1 Run a test container

```bash
docker run hello-world
```

This one command:
1. looks for the image `hello-world` on your laptop, does not find it, and **pulls** (downloads) it from Docker Hub;
2. creates a **container** from the image;
3. runs it. The program prints a message and exits, so the container stops.

You should see `Hello from Docker!`.

> **Screenshot for Part A (A1-a):** the terminal showing `docker run hello-world` and its output.

If it fails with *"Cannot connect to the Docker daemon"*, Docker Desktop is not running. Open it and wait for "Engine running".

## 3.2 Images and containers on your laptop

```bash
docker images        # images you have downloaded or built
docker ps            # containers that are running right now
docker ps -a         # ALL containers, including stopped ones
```

`docker ps` is probably empty: `hello-world` already finished. `docker ps -a` shows it with a status like `Exited (0) 2 minutes ago`. **Exit code 0 means success.** Any other number means the program failed. This matters in Part B, where Compose uses your loader's exit code to decide whether to start the dashboard.

Every image has a repository name, a **tag** (a version label; `latest` if you do not give one), an ID and a size.

## 3.3 Inspect and manage containers

```bash
docker logs <container_id_or_name>      # everything the container printed
docker inspect <container_id_or_name>   # full JSON details: settings, network, volumes
docker stats                            # live CPU and memory use (Ctrl+C to quit)

docker stop <name>        # stop a running container
docker start <name>       # start a stopped one again
docker rm <name>          # delete a stopped container
docker rmi <image>        # delete an image
docker container prune    # delete ALL stopped containers
docker image prune        # delete images no container uses
```

You only need to type the first few characters of an ID, for example `docker logs 3fa`.

> **Careful:** `docker system prune` deletes every stopped container, unused network and dangling image at once. Never run it with `--volumes` unless you mean to delete your databases too.

## 3.4 A Linux computer inside your laptop

```bash
docker run -it --name myubuntu ubuntu
```

`-it` means *interactive* plus a *terminal*: you get a shell inside an Ubuntu container. `--name` gives it a name instead of a random one like `quirky_turing`. Try:

```bash
whoami        # root
pwd
uname -a      # a Linux kernel, even on a Mac or Windows laptop
cat /etc/os-release
apt update && apt install -y curl
exit
```

Anything you install here stays inside this container and never touches your laptop. Start the same container again later with `docker start -ai myubuntu`, and delete it with `docker rm myubuntu`.

> **Screenshot for Part A (A1-b):** `docker ps -a` showing at least `hello-world` and `myubuntu`.

# 4. Build your first image

So far you ran images someone else built. Now you build your own.

## 4.1 Make a project folder

Create a folder called `my_docker_app` (anywhere outside your course repo is fine) with two files.

**`app.py`** (put your own name in the message; your screenshot must show it):

```python
print("Hello from inside a Docker container built by YOUR NAME")
```

**`requirements.txt`**, empty for now except a comment:

```text
# Python packages go here, one per line
```

## 4.2 Write the Dockerfile

Create a file named exactly `Dockerfile` (capital D, **no extension**; on Windows make sure it did not become `Dockerfile.txt`):

```dockerfile
# Start from an official image that already has Python
FROM python:3.12-slim

# Every command after this runs inside /app in the image
WORKDIR /app

# Copy the package list first and install it
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Then copy your code
COPY app.py .

# What the container runs when it starts
CMD ["python", "app.py"]
```

| Line | What it does |
|---|---|
| `FROM` | The base image to start from (Python 3.12 on a small Debian Linux) |
| `WORKDIR` | Creates and moves into a folder inside the image |
| `COPY` | Copies files from your laptop's folder into the image |
| `RUN` | Runs a command **while building** the image (installing packages) |
| `CMD` | The command the container runs **when it starts** |

**Why copy `requirements.txt` before `app.py`?** Docker builds images in **layers** and reuses (caches) every layer whose inputs did not change. With this order, editing `app.py` only rebuilds the last `COPY`; the slow `pip install` layer is reused. If you write `COPY . .` before `pip install`, every code edit reinstalls every package. In Part B, where your images install pandas, SQLAlchemy and Streamlit, that is the difference between a 2 second and a 2 minute rebuild.

## 4.3 Build and run

From inside `my_docker_app`:

```bash
docker build -t my-python-app .
docker images
docker run --rm my-python-app
```

- `-t my-python-app` gives the image a name (tag).
- The final `.` means "the Dockerfile and files are in this folder" (the **build context**).
- `--rm` deletes the container as soon as it exits, so stopped containers do not pile up.

> **Screenshot for Part A (A1-c):** `docker images` showing `my-python-app`, and the output of `docker run --rm my-python-app` with your name.

## 4.4 Add a real package

Change `app.py`:

```python
import requests

print("Hello from YOUR NAME. Making a test request...")
response = requests.get("https://httpbin.org/get", timeout=10)
print(response.status_code, response.json()["headers"]["User-Agent"])
```

Add `requests` to `requirements.txt`, then **run the image without rebuilding first**:

```bash
docker run --rm my-python-app
```

You still get the old message. **A container runs the code that was copied into the image at build time, not the files on your laptop now.** Rebuild, then run again:

```bash
docker build -t my-python-app .
docker run --rm my-python-app
```

Remember this. "I changed my code and nothing happened" is the most common Docker question in every class, and the answer is almost always "you did not rebuild." With Compose, `docker compose up --build` rebuilds for you.

Further reading: [Dockerfile reference](https://docs.docker.com/reference/dockerfile/), [Building best practices](https://docs.docker.com/build/building/best-practices/)

# 5. Docker Compose: a database and an app together

A real pipeline has several programs: a database, a loader, a dashboard. Starting each with a long `docker run ...` command is painful. **Docker Compose** describes all of them in one file, `docker-compose.yml`, and starts them together.

| Compose word | Meaning |
|---|---|
| **service** | One container in the stack (here: `db` and `app`) |
| **volume** | Storage that lives outside the container, so data survives when the container is deleted |
| **network** | Compose puts all services on one private network where each can reach the others **by service name** |
| **`.env`** | A file of settings (like passwords) that Compose reads and puts into the YAML wherever you write `${NAME}` |

## 5.1 The quotes app

You will build a two-service app: **PostgreSQL** and a small **Python script** that asks you for a quote, saves it, and prints every quote stored so far. Make a folder called `quotes_app` and create these five files in it. Type them (or copy them from this notebook); typing them once is how you learn what each line does.

```
quotes_app/
├── .env                 settings and passwords
├── docker-compose.yml   the two services
├── app.py               the Python script
├── requirements.txt     its packages
└── Dockerfile           how to build the app image
```

> Files that start with a dot are hidden by default. On macOS press **Cmd + Shift + .** in Finder to show them; in VS Code they are always visible.

**`.env`**

```env
POSTGRES_USER=postgres
POSTGRES_PASSWORD=admin
POSTGRES_DB=demo
DATABASE_URL=postgresql://user:password@db:5432/demo
```

**`docker-compose.yml`**

```yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_USER: ${POSTGRES_USER}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
      POSTGRES_DB: ${POSTGRES_DB}
    volumes:
      - db_data:/var/lib/postgresql/data

  app:
    build: .
    depends_on:
      - db
    environment:
      DATABASE_URL: ${DATABASE_URL}

volumes:
  db_data:
```

The official `postgres` image reads `POSTGRES_USER`, `POSTGRES_PASSWORD` and `POSTGRES_DB` the **first time** it starts and creates that user and database. `db_data` is a **named volume**: Docker keeps the database files there, outside the container.

> You may see old tutorials start this file with `version: '3.9'`. That line is obsolete; current Compose ignores it and prints a warning. Leave it out.

**`app.py`**

```python
import os
from sqlalchemy import create_engine, text

print("Welcome! Let's add your quote to the PostgreSQL database.")
quote = input("Enter your favorite quote: ")
db_url = os.getenv("DATABASE_URL")
engine = create_engine(db_url)

with engine.connect() as conn:
    conn.execute(text("CREATE TABLE IF NOT EXISTS quotes (id SERIAL PRIMARY KEY, message TEXT);"))
    conn.execute(text("INSERT INTO quotes (message) VALUES (:message);"), {"message": quote})
    conn.commit()
    result = conn.execute(text("SELECT * FROM quotes ORDER BY id"))
    print("\nAll quotes in the PostgreSQL database:")
    for row in result:
        print(f"{row.id}: {row.message}")
```

**`requirements.txt`**

```text
sqlalchemy==2.0.36
psycopg2-binary==2.9.10
```

**`Dockerfile`**

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
CMD ["python", "app.py"]
```

## 5.2 Build and run it

From inside `quotes_app`:

```bash
docker compose build
docker compose run app
```

**`run`, not `up`.** The script waits for you to type. `docker compose up` starts services in the background of a shared log and does **not** connect your keyboard to them, so `input()` fails. `docker compose run app` starts `app` (and `db`, because of `depends_on`) and connects your keyboard to it.

**The first run will fail, and that is on purpose.** Do not ask anyone for the fix yet. Read the last lines of the error: they name the exact problem. Compare it with your `.env`, fix the file, and run `docker compose run app` again.

> **Screenshot for Part A (A2-a):** the error from the failed run, with the line that tells you what is wrong visible.

> **Very first run only:** if you instead see *"Connection refused"*, Postgres was still starting when your script tried to connect. `depends_on` only waits for the `db` container to **start**, not for Postgres to be **ready**. Wait a few seconds and run again. In Part B you fix this properly with a healthcheck (Section 6.2).

Once it works, add a quote that **includes your name**, then add a second quote:

> **Screenshot for Part A (A2-b):** a successful run showing your quote with your name and the list of stored quotes.

## 5.3 Stopping, and what happens to the data

```bash
docker compose down        # stop and delete the containers and the network
docker compose down -v     # ...and ALSO delete the volumes (the database files)
```

Try both. After `down`, run the app again and count your quotes. Then `down -v`, run again, and count. You need what you observed for Part A, question A3.

## 5.4 Why the `.env` file

- Passwords stay out of your code and out of `docker-compose.yml`.
- You change a setting in one place without touching code.
- Each person (or each server) can have different values with the same code.

The rule used in industry: commit a **`.env.sample`** with the variable names and fake values, and **never commit `.env`**. Add `.env` to your `.gitignore`.

> **Part A, A4 (extension):** now do the extension described in the Instructions, in this same `quotes_app` folder.

# 6. Part B: running your Assignment 5 code in containers

In Assignment 5 your pipeline only runs on your laptop, with steps done by hand. In Part B you package the same work so that **one command, `docker compose up --build`, runs all of it on any machine**, starting from an empty database.

## 6.1 What you build

| Service | What it runs | Starts when |
|---|---|---|
| `db` | `postgres:16`; creates your 7 tables itself from your Assignment 4 schema file | first |
| `loader` | your Assignment 5 loader as a script, `load.py`; loads the CSV once, prints counts, **exits** | `db` is ready |
| `app` | your Assignment 5 `app.py` at http://localhost:8501 | `loader` finished without errors |

```
A6_part_b/
├── docker-compose.yml
├── .env.sample              submitted (fake password)    .env  NOT submitted
├── db/init/01_schema.sql    your Assignment 4 EmployeeDB_schema.sql
├── data/Cleaned_data.csv    NOT submitted
├── loader/   Dockerfile, requirements.txt, load.py
└── app/      Dockerfile, requirements.txt, app.py
```

Each Dockerfile follows Section 4 (`FROM python:3.12-slim`, install `requirements.txt`, copy the script, `CMD`). In the compose file, `build: ./loader` builds that service from its folder.

## 6.2 What your compose file needs

**1. The database creates its own tables.** Postgres runs every `.sql` file in `/docker-entrypoint-initdb.d/` the first time it starts on an **empty** volume. Mount your `db/init` folder there, next to the data volume:

```yaml
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./db/init:/docker-entrypoint-initdb.d:ro
```

Use `DB_NAME=EmployeeDB` and `DB_USER=postgres`. Remove any `CREATE DATABASE` or `\connect` lines from the schema file. Because the scripts only run on an empty volume, editing the schema later changes nothing until you run `docker compose down -v`.

**2. Start order.** Plain `depends_on` (Section 5) only waits for a container to *start*. Add a healthcheck to `db` and conditions to the other two:

```yaml
  db:
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U $${POSTGRES_USER} -d $${POSTGRES_DB}"]
      interval: 3s
      retries: 20
  loader:
    depends_on:
      db: { condition: service_healthy }
  app:
    depends_on:
      loader: { condition: service_completed_successfully }
```

`service_completed_successfully` means the loader exited with code 0 (Section 3.2), so a failed load must call `sys.exit(1)`.

**3. Settings from `.env`, and the host is `db`.** No password in any `.py`, Dockerfile or YAML file. Pass settings in with `environment:` (`DB_HOST: db`, `DB_PASSWORD: ${DB_PASSWORD}`, and so on) and read them in Python with `os.environ["DB_PASSWORD"]`. Inside a container `localhost` means that container itself; the database is reachable by its service name, `db`.

**4. Data in, dashboard out.** Give `loader` the volume `./data:/data:ro` and read `/data/Cleaned_data.csv`. Give `app` the port `"8501:8501"` and start Streamlit with `--server.address=0.0.0.0` so your browser can reach it. Give `db` no `ports` entry; it does not need one, and your Assignment 4 PostgreSQL already uses 5432.

## 6.3 Turning your Assignment 5 code into scripts

Four small changes. Your loader logic and your dashboard stay the same.

**Step 1. Move the loader from the notebook into `load.py`**

- A container cannot run notebook cells. It runs one `.py` file, top to bottom.
- Copy only the cells the load needs: read the CSV, clean, split rejects, insert, count.
- Leave out cells you used only to look at data (`.head()`, charts).
- Put the steps inside a `main()` function:

```python
import os, sys
import pandas as pd
import psycopg2

# settings (Step 2)

def main():
    # 1. read and clean Cleaned_data.csv        <- your Assignment 5 cells
    # 2. split good rows and rejects            <- your Assignment 5 cells
    # 3. insert into the 7 tables               <- your Assignment 5 cells
    # 4. print before / after / added counts    <- your reconciliation cell
    ...

# ending (Step 3)
```

**Step 2. Read the password and host from settings, not from the code**

- Do this in both `load.py` and `app.py`. It is the only change `app.py` needs.
- Before (Assignment 5):

```python
DB_HOST = "localhost"
DB_PASSWORD = "admin"
```

- After (Assignment 6):

```python
DB_HOST = os.environ["DB_HOST"]          # Docker fills in "db"
DB_PASSWORD = os.environ["DB_PASSWORD"]  # Docker fills it in from .env
```

- Do the same for `DB_NAME`, `DB_USER` and `DB_PORT`.
- Read the CSV from `/data/Cleaned_data.csv`, the folder Docker mounts in.

**Step 3. Make a failed load stop everything**

- Put this at the bottom of `load.py`:

```python
if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"LOAD FAILED: {exc}")
        sys.exit(1)
```

- `sys.exit(1)` tells Docker the load failed, so the dashboard does not start on an empty database.

**Step 4. List the packages in `requirements.txt`**

- A container starts with plain Python only. Each folder needs its own `requirements.txt`.
- Use the versions you used in Assignment 5 (`pip show streamlit` shows the version).
- Example for `app/requirements.txt`:

```text
streamlit==1.50.0
pandas==2.2.3
SQLAlchemy==2.0.36
psycopg2-binary==2.9.10
plotly==5.24.1
```

- A missing package shows up as `ModuleNotFoundError` in that service's log.

Want to see a complete working version? `seed.py` and `web/app.py` in the optional example notebook follow these four steps.

## 6.4 Build and test

Add one service at a time, and get it working before adding the next:

```bash
docker compose up db                                          # then, in a second terminal:
docker compose exec db psql -U postgres -d EmployeeDB -c "\dt" # 7 tables?
docker compose up --build loader                              # counts look right?
docker compose up --build                                     # dashboard at localhost:8501
docker compose up                                             # again: every "added" must be 0
docker compose down -v && docker compose up --build           # full reset from nothing
```

If the full reset works with no manual steps, your stack will run on anyone's machine. That is the standard for Part B and for the term project.

**For Part C1 (pgAdmin):** add `ports: - "5433:5432"` to `db`, then connect pgAdmin to host `localhost`, port `5433`.

Stuck on how the pieces fit together? The optional `Assignment06_Example_Three_Services.ipynb` shows the same setup working on toy data.

# 7. Troubleshooting: read the log first

Almost every problem is answered in the last 10 lines of one service's log: `docker compose logs <service>`.

| You see | Cause | Fix |
|---|---|---|
| `Cannot connect to the Docker daemon` | Docker Desktop is not running | Open it and wait for "Engine running" |
| `password authentication failed for user ...` | The user or password the app sends does not match what Postgres was created with | Make the values in `.env` agree. If you changed the password **after** the volume was created, Postgres still has the old one: `docker compose down -v` |
| `connection refused` or `could not translate host name` | Wrong host (`localhost` instead of `db`), a typo in the service name, or the app started before Postgres was ready | Use `db`; add the healthcheck and `condition: service_healthy` |
| `relation "employeedetails" does not exist` | The schema did not run: the init file is in the wrong folder, not named `.sql`, or the volume already existed | Check the mount path, then `docker compose down -v` |
| I changed my code and nothing happened | The image still has the old code | `docker compose up --build` |
| I changed `01_schema.sql` and nothing happened | Init scripts only run on an empty volume | `docker compose down -v` (this deletes the data) |
| `ModuleNotFoundError: No module named ...` | Package missing from that service's `requirements.txt` | Add it, rebuild |
| `FileNotFoundError: ... Cleaned_data.csv` | The CSV is not in `./data`, or the path in the code is not the path inside the container | Put it in `./data`; read it from `/data/...` |
| `Bind for 0.0.0.0:8501 failed: port is already allocated` | Another program (often a dashboard you left running) uses that laptop port | Stop it, or map another laptop port: `"8502:8501"` |
| `service "loader" didn't complete successfully: exit 1` | `load.py` crashed, so `app` was not started | `docker compose logs loader` |
| The browser cannot reach `localhost:8501` but the log looks fine | Streamlit listens only inside the container | Add `--server.address=0.0.0.0` to the `CMD` |
| `the attribute version is obsolete` | Old `version:` line in the compose file | Delete the line (a warning only) |
| `docker-compose: command not found` | The old hyphen tool is not installed | Use `docker compose` |
| Windows: `exec ... no such file or directory` for a `.sh` file | Windows line endings (CRLF) | Save the file with LF endings (bottom right corner of VS Code) |

# 8. Command cheat sheet

| Command | What it does |
|---|---|
| `docker build -t name .` | Build an image from the Dockerfile in this folder |
| `docker run --rm name` | Run a container from an image, delete it when it exits |
| `docker run -it name` | Run interactively with a terminal |
| `docker ps` / `docker ps -a` | Running containers / all containers |
| `docker images` | Images on this laptop |
| `docker logs <c>` | What a container printed |
| `docker compose up --build` | Rebuild images if needed, start every service |
| `docker compose up -d` | Start in the background |
| `docker compose run <service>` | Run one service with your keyboard attached (for `input()`) |
| `docker compose ps -a` | Status of every service, including exited ones |
| `docker compose logs <service>` | One service's output |
| `docker compose exec db psql -U postgres -d EmployeeDB` | Open a SQL prompt inside the running database |
| `docker compose down` | Stop and delete containers and network; **keep** volumes |
| `docker compose down -v` | Also delete volumes: **the database is gone** |

## Further reading

- [Docker Compose file reference](https://docs.docker.com/reference/compose-file/)
- [Control startup order in Compose](https://docs.docker.com/compose/how-tos/startup-order/) (healthchecks and `depends_on` conditions)
- [The official postgres image](https://hub.docker.com/_/postgres) (see "Initialization scripts")
- [Play with Docker](https://labs.play-with-docker.com/) (a browser sandbox if your laptop cannot run Docker today)